# Modeling - ATP Tennis Match Predictor

This notebook trains and compares models to predict ATP match winners, using the feature set built in `02_feature_engineering.ipynb`. Following the approach: establish a naive baseline first, then increase model complexity (Logistic Regression -> Random Forest -> XGBoost), evaluated on a time-based train/test split so the model is only ever tested on matches that happened after its training data.

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, roc_auc_score)
from xgboost import XGBClassifier

df = pd.read_csv('data/processed/atp_matches_features.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.shape

(68274, 34)

### Selecting features and target

The model needs a clean numeric matrix (X) and the target it's predicting (y). Only the engineered difference features and surface dummies go in, not raw identity columns like player names, which carry no generalisable signal.

In [11]:
surface_cols = [c for c in df.columns if c.startswith('Surface_')]

feature_cols = ['rank_diff', 'points_diff', 'points_data_missing',
                'career_win_rate_diff', 'surface_win_rate_diff',
                'recent_form_diff'] + surface_cols

X = df[feature_cols]
y = df['player_1_won']

print(X.shape, y.shape)
X.head()


(68274, 9) (68274,)


,rank_diff,points_diff,points_data_missing,career_win_rate_diff,surface_win_rate_diff,recent_form_diff,Surface_Clay,Surface_Grass,Surface_Hard
0,47,0,1,0.0,0.0,0.0,False,False,True
1,42,0,1,0.0,0.0,0.0,False,False,True
2,-8,0,1,-0.5,-0.5,-0.5,False,False,True
3,49,0,1,0.0,0.0,0.0,False,False,True
4,-171,0,1,0.0,0.0,0.0,False,False,True


### Time-based train/test split

Matches are split chronologically: train on earlier matches, test on the most recent ones, rather than a random split. This mirrors reality, a real prediction only ever has access to the past, never future results, so testing must respect that same constraint.

In [12]:
df_sorted = df.sort_values('Date').reset_index(drop=True)
X = df_sorted[feature_cols]
y = df_sorted['player_1_won']

split_date = df_sorted['Date'].quantile(0.8, interpolation='nearest')
train_mask = df_sorted['Date'] < split_date
test_mask = df_sorted['Date'] >= split_date

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train: {X_train.shape[0]} matches ({df_sorted['Date'][train_mask].min().date()} to {df_sorted['Date'][train_mask].max().date()})")
print(f"Test: {X_test.shape[0]} matches ({df_sorted['Date'][test_mask].min().date()} to {df_sorted['Date'][test_mask].max().date()})")


Train: 54612 matches (2000-01-03 to 2021-03-28)
Test: 13662 matches (2021-03-29 to 2026-07-19)


## Baseline models

Before any real models, two naive baselines set the bar to beat. If a trained model can't outperform these, it isn't actually learning anything useful.

In [13]:
majority_baseline = DummyClassifier(strategy='most_frequent')
majority_baseline.fit(X_train, y_train)
majority_acc = majority_baseline.score(X_test, y_test)

rank_baseline_preds = (X_test['rank_diff'] < 0).astype(int)
rank_baseline_acc = accuracy_score(y_test, rank_baseline_preds)

print(f"Majority-class baseline accuracy: {majority_acc:.4f}")
print(f"'Better rank always wins' baseline accuracy: {rank_baseline_acc:.4f}")

Majority-class baseline accuracy: 0.5000
'Better rank always wins' baseline accuracy: 0.6420


## Model 1 - Logistic Regression

The simplest real model, and the most interpretable. Its coefficients show which features push predictions toward Player 1 or Player 2. Features are scaled first, since logistic regression is sensitive to features being on very different numeric scales (rank_diff can be in the thousands, while win-rate diffs range from -1 to 1).

In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

log_reg_preds = log_reg.predict(X_test_scaled)
log_reg_proba = log_reg.predict_proba(X_test_scaled)[:,1]
log_reg_acc = accuracy_score(y_test, log_reg_preds)

print(f"Logistic Regression accuracy: {log_reg_acc:.4f}")
print(classification_report(y_test, log_reg_preds))

Logistic Regression accuracy: 0.6484
              precision    recall  f1-score   support

           0       0.65      0.65      0.65      6831
           1       0.65      0.65      0.65      6831

    accuracy                           0.65     13662
   macro avg       0.65      0.65      0.65     13662
weighted avg       0.65      0.65      0.65     13662



### Model 2 - Random Forest

An ensemble of decision trees. Generally stronger than a single linear model on tabular data, less sensitive to feature scaling, and provides a feature imporance ranking as a byproduct.

In [15]:
rf = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:,1]
rf_acc = accuracy_score(y_test, rf_preds)

print(f"Random Forest accuracy: {rf_acc:.4f}")
print(classification_report(y_test, rf_preds))

Random Forest accuracy: 0.6465
              precision    recall  f1-score   support

           0       0.65      0.63      0.64      6831
           1       0.64      0.66      0.65      6831

    accuracy                           0.65     13662
   macro avg       0.65      0.65      0.65     13662
weighted avg       0.65      0.65      0.65     13662

